# 05 — Forecasting

Forecasts monthly revenue forward using Holt-Winters exponential smoothing.

**Prerequisite**: this notebook needs `statsmodels`, which isn't part of the
PySpark/Delta stack. Running this once in the `pyspark-env` before starting:

```
pip install statsmodels
```

**Why pandas here and not pure PySpark**: `gold_monthly_revenue` is a tiny
table (~20-24 rows). Forecasting a single time series doesn't need
distributed computing, and PySpark has no built-in forecasting library.
So, I pull it into pandas, forecast there, then write the result back to
Delta so Power BI can read it like any other Gold table.

In [1]:
from spark_utils import get_spark_session, stop_spark_session

import pandas as pd                                            # for the small in-memory time series and forecasting step
import numpy as np                                             # for percentile calculations on the simulated forecast paths
from statsmodels.tsa.holtwinters import ExponentialSmoothing   # the forecasting model itself

# Start a Spark session tagged for this notebook's job
spark = get_spark_session("Forecasting")

# Where Gold tables live (written by 04_gold_kpis.ipynb)
GOLD_DIR = "../delta/gold"

D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
# Read gold_monthly_revenue and pull it into pandas - it's small enough that distributed processing isn't needed for the rest of this notebook.

monthly_revenue_spark = spark.read.format("delta").load(f"{GOLD_DIR}/gold_monthly_revenue")

monthly_revenue_pd = monthly_revenue_spark.toPandas()             # convert to a pandas DataFrame

# Building a proper first-of-month date column and sorting chronologically - the Gold table has separate year/month columns, which pandas time-series
# tools need combined into an actual date.
monthly_revenue_pd["period_date"] = pd.to_datetime(
    monthly_revenue_pd["year"].astype(str) + "-" + monthly_revenue_pd["month"].astype(str) + "-01"
)
monthly_revenue_pd = monthly_revenue_pd.sort_values("period_date").reset_index(drop=True)

print(monthly_revenue_pd[["period_date", "order_count", "total_revenue"]])

D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


   period_date  order_count  total_revenue
0   2016-09-01            3         267.36
1   2016-10-01          308       49507.66
2   2016-12-01            1          10.90
3   2017-01-01          789      120312.87
4   2017-02-01         1733      247303.02
5   2017-03-01         2641      374344.30
6   2017-04-01         2391      359927.23
7   2017-05-01         3660      506071.14
8   2017-06-01         3217      433038.60
9   2017-07-01         3969      498031.48
10  2017-08-01         4293      573971.68
11  2017-09-01         4243      624401.69
12  2017-10-01         4568      664219.43
13  2017-11-01         7451     1010271.37
14  2017-12-01         5624      743914.17
15  2018-01-01         7220      950030.36
16  2018-02-01         6694      844178.71
17  2018-03-01         7188      983213.44
18  2018-04-01         6934      996647.75
19  2018-05-01         6853      996517.68
20  2018-06-01         6160      865124.31
21  2018-07-01         6273      895507.22
22  2018-08

In [9]:
# --- Trim unstable ramp-up / incomplete months ---

# Olist's marketplace was just starting up in its earliest months, and its
# most recent month in this dataset is often a partial month (data was
# extracted mid-month). Both distort what a forecasting model learns as
# "normal" trend, so we drop any month with a very low order_count.
#
# MIN_ORDER_COUNT is intentionally a variable you can tune: look at the
# order_count column printed above, and adjust this threshold so it drops
# only the genuinely abnormal early/trailing months, not real slow months.

MIN_ORDER_COUNT = 50   # adjust based on what you see printed above

months_dropped = monthly_revenue_pd[monthly_revenue_pd["order_count"] < MIN_ORDER_COUNT]
print(f"Dropping {len(months_dropped)} month(s) below the order_count threshold:")
print(months_dropped[["period_date", "order_count"]])

trimmed_pd = monthly_revenue_pd[monthly_revenue_pd["order_count"] >= MIN_ORDER_COUNT].reset_index(drop=True)
print(f"\n{len(trimmed_pd)} months remaining after the order_count filter")

# --- Gap check ---

# A simple order_count threshold can drop a month sitting in the MIDDLE of
# the remaining range (not just at the start/end), leaving a hole once the
# series is forced onto a continuous monthly frequency later. A hole like
# that silently breaks the model fit (it shows up as NaN everywhere).
# If any gap is found, we trim the series to start right after the LAST
# gap month, so what's left is guaranteed to be fully contiguous.

full_range = pd.date_range(trimmed_pd["period_date"].min(), trimmed_pd["period_date"].max(), freq="MS")
missing_months = full_range.difference(trimmed_pd["period_date"])

if len(missing_months) > 0:
    last_gap = missing_months.max()
    print(f"\nGap month(s) found inside the trimmed range: {list(missing_months.date)}")
    print(f"Trimming series to start after the last gap ({last_gap.date()}) to keep it contiguous.")
    trimmed_pd = trimmed_pd[trimmed_pd["period_date"] > last_gap].reset_index(drop=True)
else:
    print("\nNo internal gaps - trimmed series is contiguous.")

min_d = trimmed_pd["period_date"].min().date()
max_d = trimmed_pd["period_date"].max().date()
print(f"\nFinal series: {len(trimmed_pd)} months, from {min_d} to {max_d}")

Dropping 3 month(s) below the order_count threshold:
   period_date  order_count
0   2016-09-01            3
2   2016-12-01            1
23  2018-09-01            1

21 months remaining after the order_count filter

Gap month(s) found inside the trimmed range: [datetime.date(2016, 11, 1), datetime.date(2016, 12, 1)]
Trimming series to start after the last gap (2016-12-01) to keep it contiguous.

Final series: 20 months, from 2017-01-01 to 2018-08-01


In [10]:
# --- Fit the Holt-Winters model ---
# trend="add": captures a linear-ish upward/downward trend
# seasonal=None: with under ~24 clean months, there aren't enough full
# yearly cycles to reliably estimate a seasonal component - forcing one
# in would just be overfitting noise, not a real seasonal pattern.

revenue_series = trimmed_pd.set_index("period_date")["total_revenue"]  # pandas Series indexed by date, required by statsmodels

# statsmodels needs the index to carry an explicit frequency (e.g. "MS" =
# month start) so it knows how to extend the index into future months
# during forecasting/simulation. Without this, .simulate() fails with
# "No supported index is available."
revenue_series = revenue_series.asfreq("MS")

# asfreq() will insert a NaN row for any month missing from the trimmed
# data (a "gap" in the middle of the series, as opposed to the ramp-up
# months we already trimmed off the edges). Check for that before fitting -
# a gap would silently break the model otherwise.
if revenue_series.isna().any():
    print("WARNING: gap(s) found in the trimmed monthly series:")
    print(revenue_series[revenue_series.isna()])
else:
    print("No gaps in the trimmed monthly series - safe to fit.")

model = ExponentialSmoothing(
    revenue_series,
    trend="add",              # additive trend component
    seasonal=None,               # no seasonal component - insufficient history to estimate one reliably
    initialization_method="estimated",  # let statsmodels estimate starting level/trend from the data itself
)

fitted_model = model.fit()   # fit the model to the trimmed historical series

print(fitted_model.summary())

No gaps in the trimmed monthly series - safe to fit.
                       ExponentialSmoothing Model Results                       
Dep. Variable:            total_revenue   No. Observations:                   20
Model:             ExponentialSmoothing   SSE                   237849802177.871
Optimized:                         True   AIC                            471.983
Trend:                         Additive   BIC                            475.966
Seasonal:                          None   AICC                           478.445
Seasonal Periods:                  None   Date:                 Thu, 17 Sep 2026
Box-Cox:                          False   Time:                         12:51:55
Box-Cox Coeff.:                    None                                         
                       coeff                 code              optimized      
------------------------------------------------------------------------------
smoothing_level            0.4110054                alpha   

In [11]:
# --- Forecast forward, with confidence bounds via simulation ---
# ExponentialSmoothing's plain .forecast() only gives a single point estimate
# per future month. To get a confidence interval, we simulate many possible
# future paths (accounting for the model's estimated error) and take
# percentiles across those simulations at each future month.

FORECAST_MONTHS = 6          # how many months ahead to forecast
N_SIMULATIONS = 1000           # number of simulated future paths to generate

simulations = fitted_model.simulate(
    nsimulations=FORECAST_MONTHS,   # simulate this many months into the future
    repetitions=N_SIMULATIONS,        # run that many independent simulated paths
    error="add",                        # additive error, matching the additive trend above
)
# simulations is a DataFrame: each column is one simulated future path,
# each row is one future month.

forecast_point = simulations.mean(axis=1)                # point forecast = average across all simulated paths
forecast_lower = simulations.quantile(0.10, axis=1)         # 10th percentile = lower bound of the confidence range
forecast_upper = simulations.quantile(0.90, axis=1)          # 90th percentile = upper bound of the confidence range

print(pd.DataFrame({
    "forecast": forecast_point,
    "lower_bound": forecast_lower,
    "upper_bound": forecast_upper,
}))

                 forecast    lower_bound   upper_bound
2018-09-01  868716.279496  711748.143485  1.027212e+06
2018-10-01  838064.375788  637797.226331  1.036603e+06
2018-11-01  809764.193515  534518.036403  1.094750e+06
2018-12-01  783629.852643  414827.480670  1.151298e+06
2019-01-01  759971.287980  273543.831206  1.229311e+06
2019-02-01  726274.383945  130773.297098  1.335986e+06


In [12]:
# --- Combine actuals + forecast into one table ---
# One row per month, whether actual history or a future forecast, so
# Power BI can plot both on a single continuous line from one data source.

actual_part = pd.DataFrame({
    "period_date": trimmed_pd["period_date"],
    "total_revenue": trimmed_pd["total_revenue"],   # real historical revenue
    "forecast_revenue": np.nan,                        # no forecast value for actual months
    "lower_bound": np.nan,
    "upper_bound": np.nan,
    "is_forecast": False,
})

forecast_part = pd.DataFrame({
    "period_date": forecast_point.index,
    "total_revenue": np.nan,                       # no actual value exists yet for future months
    "forecast_revenue": forecast_point.values,
    "lower_bound": forecast_lower.values,
    "upper_bound": forecast_upper.values,
    "is_forecast": True,
})

gold_revenue_forecast_pd = pd.concat([actual_part, forecast_part], ignore_index=True)
gold_revenue_forecast_pd = gold_revenue_forecast_pd.sort_values("period_date").reset_index(drop=True)

print(gold_revenue_forecast_pd)

   period_date  total_revenue  forecast_revenue    lower_bound   upper_bound  \
0   2017-01-01      120312.87               NaN            NaN           NaN   
1   2017-02-01      247303.02               NaN            NaN           NaN   
2   2017-03-01      374344.30               NaN            NaN           NaN   
3   2017-04-01      359927.23               NaN            NaN           NaN   
4   2017-05-01      506071.14               NaN            NaN           NaN   
5   2017-06-01      433038.60               NaN            NaN           NaN   
6   2017-07-01      498031.48               NaN            NaN           NaN   
7   2017-08-01      573971.68               NaN            NaN           NaN   
8   2017-09-01      624401.69               NaN            NaN           NaN   
9   2017-10-01      664219.43               NaN            NaN           NaN   
10  2017-11-01     1010271.37               NaN            NaN           NaN   
11  2017-12-01      743914.17           

In [13]:
# --- Write the combined actual + forecast table back to Delta ---

gold_revenue_forecast_spark = spark.createDataFrame(gold_revenue_forecast_pd)   # convert pandas back to a Spark DataFrame

gold_revenue_forecast_spark.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/gold_revenue_forecast")

print(f"[Gold] revenue_forecast: {gold_revenue_forecast_spark.count()} rows written")

D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


[Gold] revenue_forecast: 26 rows written


In [14]:
stop_spark_session(spark)